# 12 CAGE-4 Dependency Check

Run this notebook before the CAGE-4 rollout stages. This checks the packages needed for our CPU rollout path. `ray` and `pettingzoo` are reported as optional because stages 13-15 do not use RLlib/PettingZoo training directly.


In [1]:
from pathlib import Path
import os, sys

REPO_ROOT_OVERRIDE = None


def _looks_like_repo(base: Path) -> bool:
    return (base / "pyproject.toml").exists() and (base / "src" / "cybsage_marl").exists()


def _mount_drive_if_colab():
    if not Path("/content").exists():
        return
    if Path("/content/drive/MyDrive").exists() or Path("/content/drive/My Drive").exists():
        return
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive auto-mount failed:", exc)


def _find_repo_root():
    _mount_drive_if_colab()
    candidates = []
    if REPO_ROOT_OVERRIDE:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser())
    if os.environ.get("CYBSAGE_REPO_ROOT"):
        candidates.append(Path(os.environ["CYBSAGE_REPO_ROOT"]).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([
        Path("/content/drive/MyDrive/projects/implementation/IMPLEMENTATION"),
        Path("/content/drive/My Drive/projects/implementation/IMPLEMENTATION"),
        Path("/content/drive/MyDrive/IMPLEMENTATION"),
        Path("/content/drive/My Drive/IMPLEMENTATION"),
        Path("/content/drive/MyDrive/projects/implementation/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
        Path("/content/drive/My Drive/projects/implementation/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
        Path("/Volumes/LocalDrive1/onlyscholar-projects/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
    ])

    checked = []
    seen = set()
    for base in candidates:
        base = base.expanduser()
        if str(base) in seen:
            continue
        seen.add(str(base))
        checked.append(base)
        if _looks_like_repo(base):
            return base

    # Last-resort bounded search. This catches Drive shortcuts/copies while avoiding a full Drive crawl.
    search_roots = [Path("/content/drive/MyDrive"), Path("/content/drive/My Drive"), Path("/content")]
    for root in search_roots:
        if not root.exists():
            continue
        try:
            for pyproject in root.glob("**/pyproject.toml"):
                base = pyproject.parent
                checked.append(base)
                if base.name == "IMPLEMENTATION" and _looks_like_repo(base):
                    return base
        except Exception as exc:
            checked.append(Path(f"{root} [search failed: {exc}]"))

    raise RuntimeError(
        "Could not find CybSAGE IMPLEMENTATION repo root visible to this runtime.\n\n"
        "Most common fix: in Colab, run drive.mount('/content/drive') and confirm the folder exists under "
        "/content/drive/MyDrive/projects/implementation/IMPLEMENTATION.\n\n"
        "Checked:\n" + "\n".join(map(str, checked[-80:]))
    )


REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print("Repository root:", REPO_ROOT)


Repository root: /content/drive/MyDrive/projects/implementation/IMPLEMENTATION


In [2]:
from cybsage_marl.config import load_config
from cybsage_marl.data.cyborg_adapter import cyborg_import_status
from cybsage_marl.cage4.compat import install_ray_rllib_stub

cfg = load_config()
cyborg_repo = cfg.cyborg_dir / "cage-challenge-4"
print("CybORG repo:", cyborg_repo)
print(cyborg_import_status(cyborg_repo))
print("Ray/RLlib shim active for CPU rollouts:", install_ray_rllib_stub())
print("Notebook 12 will mark ready when required CPU-rollout imports and the CAGE-4 wrapper import work.")


CybORG repo: /content/drive/MyDrive/projects/Dataset/Security/CybORG_CAGE_Challenge_4/cage-challenge-4


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_r

{'repo': '/content/drive/MyDrive/projects/Dataset/Security/CybORG_CAGE_Challenge_4/cage-challenge-4', 'path_added': True, 'import_ok': True, 'module_file': '/content/drive/MyDrive/projects/Dataset/Security/CybORG_CAGE_Challenge_4/cage-challenge-4/CybORG/__init__.py'}
Ray/RLlib shim active for CPU rollouts: True
Notebook 12 will mark ready when required CPU-rollout imports and the CAGE-4 wrapper import work.


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [3]:
!PYTHONPATH=src python scripts/run_stage.py cage4_check

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.13/dist-packages/pkg_resources/__init__.py:3148: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`

In [4]:
import json
from pathlib import Path
summary = json.loads(Path("outputs/12_cage4_dependency_check/dependency_summary.json").read_text())
print(json.dumps(summary, indent=2))
if not summary.get("ready_for_cage4_rollouts"):
    raise RuntimeError("CAGE-4 CPU rollout dependencies are not ready. Check missing_required_packages and wrapper_import_error above.")
print("Ready: run 13_cage4_baseline_rollouts.ipynb on CPU next.")


{
  "cyborg_repo": "/content/drive/MyDrive/projects/Dataset/Security/CybORG_CAGE_Challenge_4/cage-challenge-4",
  "cyborg_repo_ok": true,
  "python": "/usr/bin/python3",
  "ready_for_cage4_rollouts": true,
  "missing_required_packages": [],
  "missing_optional_packages": [
    "ray",
    "pettingzoo"
  ],
  "wrapper_import_ok": true,
  "wrapper_import_error": "",
  "ray_rllib_stub_used": true,
  "install_hint": ""
}
Ready: run 13_cage4_baseline_rollouts.ipynb on CPU next.
